# LDA Topic analysis

In [1]:
from utils.ml import data_dir, load_data, tokenize

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/21 16:49:09 WARN Utils: Your hostname, DESKTOP-UQF5BSK, resolves to a loopback address: 127.0.1.1; using 172.24.225.163 instead (on interface eth0)
26/04/21 16:49:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 16:49:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = load_data(data_dir)
tokens = tokenize(df, only_tokens=True)

In [3]:
from pyspark.ml import PipelineModel
from pyspark.ml.clustering import LDAModel
from pyspark.ml.feature import CountVectorizerModel
pp = PipelineModel.load("LDA_pipeline")
model: LDAModel = pp.stages[-1]
vectorizer: CountVectorizerModel = pp.stages[0]
topic_info = model.describeTopics(maxTermsPerTopic=15)

In [4]:

from pyspark.sql import functions as F
melted = topic_info.withColumn("zip", F.arrays_zip("termIndices", "termWeights"))
melted = melted.select("topic", F.explode("zip"))
melted = melted.select("topic", F.col("col.termIndices").alias("term"), F.col("col.termWeights").alias("weight"))
mapping_expr = F.create_map(*[x for i, w in enumerate(vectorizer.vocabulary) for x in (F.lit(i), F.lit(w))])
melted = melted.withColumn("id", mapping_expr[F.col("term")])
melted.show(5)

+-----+----+--------------------+---+
|topic|term|              weight| id|
+-----+----+--------------------+---+
|    0|   0| 0.06131722944226176| 51|
|    0|   4|0.060549569773505375| 76|
|    0|  14|0.046760120544508996|215|
|    0|  13| 0.04453268559413277|181|
|    0|  21|0.035214648891271495| 46|
+-----+----+--------------------+---+
only showing top 5 rows


In [5]:
final = melted.join(tokens, on="id", how = "left")
final.show(10)

+---+-----+----+--------------------+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+
| id|topic|term|              weight|page 1 (main category)|page 2 (clothing model)|colour|location|model photography|price|price 2|page|
+---+-----+----+--------------------+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+
| 51|    0|   0| 0.06131722944226176|                     2|                     B4|    10|       2|                1|   52|      1|   1|
| 76|    0|   4|0.060549569773505375|                     2|                    B10|     2|       4|                1|   67|      1|   1|
|215|    0|  14|0.046760120544508996|                     2|                    B24|    11|       2|                1|   57|      1|   2|
|181|    0|  13| 0.04453268559413277|                     2|                    B13|    12|       5|                2|   62|      1|   1|
| 46|    0|  21|0.0352146488912714

In [6]:
topic_profiles = final.groupBy("topic").agg(
        F.avg("price").alias("avg_price"),
        F.mode("price").alias("top_price"),
        F.median("price").alias("median_price"),
        F.stddev("price").alias("price_std"),
        F.percentile("price", 0.05).alias("hi"),
        F.percentile("price", 0.95).alias("lo"),
        F.percentile("price", 0.25).alias("q1"),
        F.percentile("price", 0.75).alias("q3"),
        F.countDistinct("colour").alias("colour_diversity"),
        F.mode("colour").alias("top_colour"),
        F.countDistinct("page 1 (main category)").alias("total_categories"),
        F.mode("page 1 (main category)").alias("top_category"),
        F.mean("page").alias("avg_page_depth"),
        F.mean(F.col("model photography") - 1).alias("photography_probability"),
        F.collect_set("page 2 (clothing model)").alias("product_set"),
        F.collect_set("page 1 (main category)").alias("category_set"),
        F.collect_set("colour").alias("color_set")
    )
topic_profiles.show()

26/04/21 16:49:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----+------------------+---------+------------+------------------+----+-----------------+----+----+----------------+----------+----------------+------------+------------------+-----------------------+--------------------+------------+--------------------+
|topic|         avg_price|top_price|median_price|         price_std|  hi|               lo|  q1|  q3|colour_diversity|top_colour|total_categories|top_category|    avg_page_depth|photography_probability|         product_set|category_set|           color_set|
+-----+------------------+---------+------------+------------------+----+-----------------+----+----+----------------+----------+----------------+------------+------------------+-----------------------+--------------------+------------+--------------------+
|    0|              50.4|       38|        52.0|10.513936329598783|38.0|63.49999999999999|38.0|57.0|               9|         2|               1|           2|1.3333333333333333|    0.13333333333333333|[B32, B14, B24, B...|   